In [11]:
import os
import numpy as np
import pandas as pd
import json
import systeme as sys
import Allocation
from simulation import simulation
import time

import pickle
import random
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, make_scorer, hamming_loss
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.base import BaseEstimator
from functools import partial

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

instance_name = "G"

if instance_name == "K0":
    test_split_scenarios = [23, 16, 7, 10, 26, 17, 6, 20, 11]
    fms_path = 'fms/3C7R5F.json'
    nombre_de_cellules = 3
    nombre_de_scenarios = 28
else :
    test_split_scenarios = [37, 13, 31, 40, 25, 14, 6, 12, 4, 7, 28, 15, 17]
    fms_path = 'fms/5C14R5F.json'
    nombre_de_cellules = 5
    nombre_de_scenarios = 40

with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
system = sys.systeme(dic)

## generation de data

### génération de data

In [12]:
verbose = 1
ds = instance_name
scenarios_path_prefixe = f"scenarios/{ds}/"
solutions_path_prefixe = f"solution/{ds}_upgraded/"
with open(fms_path, 'r') as json_file:
    dic = json.load(json_file)
system = sys.systeme(dic)
for scenario_path, solution_path in zip(sorted(os.listdir(scenarios_path_prefixe), key= lambda k : int(k[1:].split(".csv")[0])), sorted(os.listdir(solutions_path_prefixe), key= lambda k : int(k.split(".csv")[0].split("_s")[-1]))):
    if verbose > 0:
        print(f"scenario : {scenario_path}",end="\r")
    #initialisation de solution systeme et scenario
    solution = pd.read_csv(solutions_path_prefixe+solution_path, sep=";", index_col=None, header=None).iloc[:,1:-1]
    scenario = pd.DataFrame(np.nan_to_num(pd.read_csv(scenarios_path_prefixe+scenario_path,header=None, index_col=None, sep=";"), nan=0)).astype(int)
    sim = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))], saving=True)
    logs = sim.get_logs()
    [df.to_csv(f"generated_data/{ds}/single_label/singlelabel_data_sortedlabels_{scenario_path.split(".")[0]}_cell_{i+1}.csv", sep=";", index=False) for df,i in zip(logs, range(len(logs)))]

### generation de data multilabel

In [13]:
def get_voisins(df:pd.DataFrame, line:int, col:int, system:sys.systeme):
    offset = sum([len(x.ressources) for x in system.cellules[:col]])+1
    arr = np.arange(offset,offset+len(system.cellules[col].ressources))
    voisins_locaux = np.delete(arr, np.where(arr == df.iloc[line, col])[0])
    voisins = []
    for voisin in voisins_locaux:
        df_copy = df.copy()
        df_copy.iloc[line, col] = voisin
        voisins.append(df_copy)
    return voisins

def get_echanges(df:pd.DataFrame, line:int, col:int):
    arr_solution = df.values
    valeur_courante = arr_solution[line, col]

    lignes_suivantes = np.arange(line + 1, arr_solution.shape[0])
    valeurs_suivantes = arr_solution[lignes_suivantes, col]

    mask_diff = valeurs_suivantes != valeur_courante
    lignes_differentes = lignes_suivantes[mask_diff]
    valeurs_differentes = valeurs_suivantes[mask_diff]

    if len(lignes_differentes) == 0:
        return []

    echanges_array = np.repeat(arr_solution[np.newaxis, :, :], len(lignes_differentes), axis=0)
    echanges_array[:, line, col] = valeurs_differentes
    echanges_array[np.arange(len(lignes_differentes)), lignes_differentes, col] = valeur_courante

    return [pd.DataFrame(echange, columns=df.columns) for echange in echanges_array]

def upgrade_insert(system:sys.systeme, scenario:pd.DataFrame, solution:pd.DataFrame, stats:dict, verbose:int):
    symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
    symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
    symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
    produit, cellule = 0,0
    upgraded = False
    old_mct = best_mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))]).mean_completion_time()
    while cellule < solution.shape[1]:
        while produit < solution.shape[0]:
            reset = False
            solutions_generees = get_voisins(solution, line=produit, col=cellule, system=system)
            iteration = 0
            for voisin in solutions_generees:
                if verbose > 0:
                    print(f"\tline {produit+1}, col {cellule+1}/{solution.shape[1]}, iteration {iteration+1}/{len(solutions_generees)}                              ",end="\r")
                try:
                    mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, voisin) for _ in range(len(system.cellules))]).mean_completion_time()
                except Exception as e:
                    if verbose >0:
                        print("changement non autorisé : ",e)
                    iteration+=1
                    continue
                if (mct == best_mct):
                    if verbose > 1:
                        print(f"\n\t\tsymetrie produit {produit+1} cell {cellule+1} solution originale {solution.iloc[produit, cellule]} avec solution {voisin.iloc[produit, cellule]}")
                    symetries_indices[produit, cellule].append(iteration)
                    if stats is not None:
                        stats["symetries"][1,cellule] += 1
                elif (mct < best_mct):
                    if verbose > 0:
                        print(f"new best solution found, from {best_mct} to {mct}, redo all")
                    if stats is not None:
                        stats["improvements"] +=1
                    best_mct = mct
                    solution = voisin
                    reset = True
                    upgraded = True
                    if stats is not None:
                        stats["symetries"] = np.zeros_like(stats["symetries"])
                    break
                iteration+=1
            if reset : 
                cellule, produit = 0, 0
                symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
                symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
                symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
            else :
                produit+=1
        cellule+=1
        produit = 0

    return solution, upgraded, old_mct, best_mct, symetries_indices

def upgrade_swap(system:sys.systeme, scenario:pd.DataFrame, solution:pd.DataFrame, stats:dict, verbose:int):
    symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
    symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
    symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
    produit, cellule = 0,0
    upgraded = False
    old_mct = best_mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))]).mean_completion_time()
    while cellule < solution.shape[1]:
        while produit < solution.shape[0]:
            reset = False
            solutions_generees = get_echanges(solution, produit, cellule)
            iteration = 0
            for i in range(len(solutions_generees)):
                if verbose > 0:
                    print(f"\tline {produit+1}, col {cellule+1}/{solution.shape[1]}, iteration {iteration+1}/{len(solutions_generees)}                              ",end="\r")
                try:
                    mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solutions_generees[i]) for _ in range(len(system.cellules))]).mean_completion_time()
                except Exception as e:
                    if verbose > 0 :
                        print("changement non autorisé : ",e)
                    iteration+=1
                    continue
                if (mct == best_mct):
                    if verbose > 1:
                        print(f"\n\t\tsymetrie produit {produit+1} cell {cellule+1} solution originale {solution.iloc[produit, cellule]} avec solution {solutions_generees[i].iloc[produit, cellule]}")
                    symetries_indices[produit, cellule].append(iteration)
                    if stats is not None:
                        stats["symetries"][0,cellule] += 1
                elif (mct < best_mct):
                    if verbose > 0:
                        print(f"\n\t\tnew best solution found, from {best_mct} to {mct}, redo all\n")
                    if stats is not None:
                        stats["improvements"] +=1
                    best_mct = mct
                    solution = solutions_generees[i]
                    reset = True
                    upgraded = True
                    if stats is not None:
                        stats["symetries"] = np.zeros_like(stats["symetries"])
                    break
                iteration+=1

            if reset : 
                cellule, produit = 0, 0
                symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
                symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
                symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
            else :
                produit+=1
        cellule+=1
        produit = 0

    return solution, upgraded, old_mct, best_mct, symetries_indices

def alter_multilabel_dataset_symetries(system:sys.systeme, solution:pd.DataFrame, swapORinsert:int, multilabel_logs:pd.DataFrame, indices_symetrie:np.ndarray, verbose:int):
    for produit in range(indices_symetrie.shape[0]):
        for cellule in range(indices_symetrie.shape[1]):
            if swapORinsert == 0:
                solutions_generees = get_echanges(solution, produit, cellule)
            elif swapORinsert == 1:
                solutions_generees = get_voisins(solution, produit, cellule, system)
            for iteration,i in zip(indices_symetrie[produit,cellule], range(len(indices_symetrie[produit,cellule]))):
                if verbose >0:
                    print(f"\t\tline {produit+1}/{indices_symetrie.shape[0]}, col {cellule+1}/{indices_symetrie.shape[1]}, iteration {i+1}/{len(indices_symetrie[produit,cellule])}",end="\r")
                multilabel_logs[cellule].iloc[produit, solutions_generees[iteration].iloc[produit, cellule]-1 - sum([len(c.ressources) for c in system.cellules][:cellule])-len(system.cellules[cellule].ressources)] = 1  
    return multilabel_logs


In [14]:
verbose = 1
ds = instance_name
scenarios_path_prefixe = f"scenarios/{ds}/"
solutions_path_prefixe = f"solution/{ds}_best/"
upgraded_solutions_path_prefixe = f"solution/{ds}_upgraded/"

with open(fms_path, 'r') as json_file: #5C14R5F #3C7R5F
    dic = json.load(json_file)
system = sys.systeme(dic)

for scenario_path, solution_path in zip(sorted(os.listdir(scenarios_path_prefixe), key= lambda k : int(k[1:].split(".csv")[0])), sorted(os.listdir(solutions_path_prefixe), key= lambda k : int(k.split(".csv")[0].split("_s")[-1]))):
    if verbose > -1:
        print(f"\nscenario : {scenario_path}")
    #initialisation de solution systeme et scenario
    solution = pd.read_csv(upgraded_solutions_path_prefixe+solution_path, sep=";", index_col=None, header=None).iloc[:,1:-1]
    scenario = pd.DataFrame(np.nan_to_num(pd.read_csv(scenarios_path_prefixe+scenario_path, index_col=None, sep=";", header=None), nan=0)).astype(int)
    reset = True
    start_time = time.time()
    while reset:
        solution, upgraded, old_mct, new_mct, symetries_indices_swap = upgrade_swap(system, scenario, solution, None, verbose)
        if verbose > 0:
            print(f"\n\n\tswap improved ? {upgraded} " + (f"from {old_mct} to {new_mct}" if upgraded else f" with {old_mct}"))
        solution, upgraded, old_mct, new_mct, symetries_indices_insert = upgrade_insert(system, scenario, solution, None, verbose)
        if verbose > 0:
            print(f"\n\n\tinsert improved ? {upgraded} " + (f"from {old_mct} to {new_mct}" if upgraded else f" with {old_mct}"))
        reset = upgraded
    # arrivés là on est surs d'avoir la meilleure solution non ameliorable dans solution, on sauvegarde la solution
    solution.insert(0, None, [f"P{i+1}" for i in range(len(solution))])
    solution[len(solution.columns)] = 0
    solution.to_csv(upgraded_solutions_path_prefixe+solution_path, sep=";", index=False, header=False)

    # symetries
    if verbose > 0:
        print(f"\n\t meilleure solution trouvée, début traitement symétries\n")
    base_logs = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution.iloc[:,1:-1]) for _ in range(len(system.cellules))], saving=True).get_logs()
    expected_classes = [
        [0, 1],    # à changer, c'est le nombre de ressources par cellule
        [0, 1, 2], 
        [0, 1, 2, 3], 
        [0, 1, 2], 
        [0, 1]     
    ]if instance_name is "G" else [
        [0,1],
        [0,1,2],
        [0,1]
    ]

    # Transformation de base_logs
    multilabel_logs = []
    for i, base_log in enumerate(base_logs):
        selected_column = base_log.columns[-1]
        dummies = pd.get_dummies(base_log, columns=[selected_column], drop_first=False, dtype=int)

        # S'assurer que toutes les colonnes attendues sont présentes
        for cls in expected_classes[i]:
            col_name = selected_column +"_"+ str(cls)+".0"
            if col_name not in dummies.columns:
                dummies[col_name] = 0  # Ajouter la colonne si elle est absente

        # Réorganiser l'ordre des colonnes
        cols = list(base_log.columns[:-1]) + [selected_column +"_"+ str(cls)+".0" for cls in expected_classes[i]]
        dummies = dummies[cols]  
        
        new_base_log = dummies
        
        multilabel_logs.append(new_base_log)
    
    for cell_log, i in zip(multilabel_logs, range(len(multilabel_logs))): # solution opti locale sans symetries
        cell_log.to_csv(f"generated_data/{ds}/multilabel/none/multilabel_data_original_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)
    
    multilabel_logs_swap = alter_multilabel_dataset_symetries(system, solution.iloc[:,1:-1], 0, multilabel_logs, symetries_indices_swap, verbose)
    for cell_log, i in zip(multilabel_logs_swap, range(len(multilabel_logs_swap))): # avec swap seul
        cell_log.to_csv(f"generated_data/{ds}/multilabel/swap/multilabel_data_swap_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)

    multilabel_logs_insert = alter_multilabel_dataset_symetries(system, solution.iloc[:,1:-1], 1, multilabel_logs, symetries_indices_insert, verbose)  
    for cell_log, i in zip(multilabel_logs_insert, range(len(multilabel_logs_insert))): # avec insert seul
        cell_log.to_csv(f"generated_data/{ds}/multilabel/insert/multilabel_data_insert_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)


scenario : s1.csv
	line 99, col 5/5, iteration 1/1                                

	swap improved ? False  with 296.5
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 296.5

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 2/20
scenario : s2.csv
	line 99, col 5/5, iteration 1/1                                

	swap improved ? False  with 305.71
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 305.71

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 1/10
scenario : s3.csv
	line 96, col 5/5, iteration 4/4                                

	swap improved ? False  with 303.49
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 303.49

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 1/12
scenario : s4.csv
	line 99, col 5/5

In [15]:
# R1 et R2 dans la meme cellule  ---
# R1 tps setup R2 tps setup   ---
# R1 tps process R2 tps process  ---
# R1 pred_fam == R2 pred_fam 
# R1 current charge == R2 current charge


# 1 localliser les ressources identiques dans systeme pour chaque cellule
ds = instance_name
data_path_prefixe = f"generated_data/{ds}/multilabel/none/"
data_path_save = f"generated_data/{ds}/multilabel/machine_id/"
with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
system = sys.systeme(dic)
nb_symetries = {}
nb_cells = nombre_de_cellules
identiques = []
for c in system.cellules:
    ids_cell = []
    for r in c.ressources:
        ids_ress = []
        for r2 in c.ressources:
            if r is r2 or (r.processTimes == r2.processTimes and r.setupTimes == r2.setupTimes) :
                ids_ress.append(True)
            else :
                ids_ress.append(False)
        ids_cell.append(ids_ress)
    identiques.append(ids_cell)
identical_couples = [[(i+1, j+1) for i, j in np.argwhere((np.array(idd).astype(int) == 1) & np.triu(np.ones(np.array(idd).astype(int).shape, dtype=bool), k=1))] for idd in identiques]
cell_containing_identities = np.where(np.array([int(np.sum(x)/len(x)) for x in identiques]) != 1)[0]

# 2 pour chaque dataset d'un scenario et d'une cellule contenant des identités
for dataset_path in os.listdir(data_path_prefixe):
    if dataset_path.endswith("json"):
        continue
    print(f"\nscenario {dataset_path.split("_cell")[0].split("_s")[-1]} :")
    cell = int(dataset_path[:-4].split("_")[-1]) -1

    nb_symetries[f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv"] = nb_symetries.get(f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv", [0]*nb_cells)
    if cell in cell_containing_identities:
        print(f"\tcellule {cell+1} :")
        dataset = pd.read_csv(data_path_prefixe+dataset_path, sep=";", index_col=None)
        
        # 3 trouver les colonnes a verifier
        for line in range(dataset.shape[0]):
            for couple in identical_couples[cell]:
                # 4 verifier les conditions
                if dataset["predFamilies_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[0])][line] == dataset["predFamilies_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[1])][line]  and  dataset["Cur_Charge_comparable_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[0])][line] == dataset["Cur_Charge_comparable_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[1])][line] \
                and  ((dataset[f"Selected Resource_{couple[0]-1}.0"][line] + dataset[f"Selected Resource_{couple[1]-1}.0"][line]) == 1):
                    print("\t\tsymetrie ligne : ", line)
                    nb_symetries[f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv"][cell] += 1
                    # 5 succes, symetrie detectee rajouter les annotations
                    dataset.loc[line, f"Selected Resource_{couple[0]-1}.0"] = 1
                    dataset.loc[line, f"Selected Resource_{couple[1]-1}.0"] = 1

        # 6 sauvegarder le dataframe
        dataset.to_csv(data_path_save+dataset_path, sep=";", index=None)

#   2   3   2



scenario 10 :
	cellule 1 :
		symetrie ligne :  0

scenario 10 :

scenario 10 :

scenario 10 :
	cellule 4 :
		symetrie ligne :  0
		symetrie ligne :  0
		symetrie ligne :  1
		symetrie ligne :  1
		symetrie ligne :  2
		symetrie ligne :  4
		symetrie ligne :  4
		symetrie ligne :  5
		symetrie ligne :  7
		symetrie ligne :  8
		symetrie ligne :  9
		symetrie ligne :  10
		symetrie ligne :  10
		symetrie ligne :  11
		symetrie ligne :  12
		symetrie ligne :  13
		symetrie ligne :  13
		symetrie ligne :  14
		symetrie ligne :  17
		symetrie ligne :  17
		symetrie ligne :  18
		symetrie ligne :  20
		symetrie ligne :  21
		symetrie ligne :  24
		symetrie ligne :  24
		symetrie ligne :  25
		symetrie ligne :  28
		symetrie ligne :  30
		symetrie ligne :  30
		symetrie ligne :  31
		symetrie ligne :  32
		symetrie ligne :  32
		symetrie ligne :  33
		symetrie ligne :  33
		symetrie ligne :  34
		symetrie ligne :  34
		symetrie ligne :  35
		symetrie ligne :  37
		symetrie ligne :  37
		syme


scenario 14 :

scenario 14 :

scenario 14 :
	cellule 4 :
		symetrie ligne :  0
		symetrie ligne :  0
		symetrie ligne :  1
		symetrie ligne :  3
		symetrie ligne :  3
		symetrie ligne :  4
		symetrie ligne :  4
		symetrie ligne :  7
		symetrie ligne :  8
		symetrie ligne :  11
		symetrie ligne :  12
		symetrie ligne :  13
		symetrie ligne :  14
		symetrie ligne :  16
		symetrie ligne :  19
		symetrie ligne :  20
		symetrie ligne :  21
		symetrie ligne :  22
		symetrie ligne :  22
		symetrie ligne :  24
		symetrie ligne :  24
		symetrie ligne :  26
		symetrie ligne :  26
		symetrie ligne :  27
		symetrie ligne :  29
		symetrie ligne :  35
		symetrie ligne :  38
		symetrie ligne :  39
		symetrie ligne :  41
		symetrie ligne :  43
		symetrie ligne :  43
		symetrie ligne :  44
		symetrie ligne :  46
		symetrie ligne :  47
		symetrie ligne :  48
		symetrie ligne :  51
		symetrie ligne :  52
		symetrie ligne :  52
		symetrie ligne :  53
		symetrie ligne :  54
		symetrie ligne :  54
		symetr

#### concatenage des multilabel en un dataset final multilabel

In [16]:
destination_path = "generated_data/"+instance_name+"/multilabel/final/"
source_none_path = "generated_data/"+instance_name+"/multilabel/none/"
source_swap_path = "generated_data/"+instance_name+"/multilabel/swap/"
source_insert_path = "generated_data/"+instance_name+"/multilabel/insert/"
source_mid_path = "generated_data/"+instance_name+"/multilabel/machine_id/"

swap_files = [source_swap_path+f for f in os.listdir(source_swap_path)]
insert_files = [source_insert_path+f for f in os.listdir(source_insert_path)]
mid_files = [source_mid_path+f for f in os.listdir(source_mid_path)]



nb_cells = nombre_de_cellules
nb_scenarios = nombre_de_scenarios

for c in range(1,1+nb_cells):
    for scen in range(1,nb_scenarios+1):
        df_swap = pd.read_csv(source_swap_path+f"multilabel_data_swap_s{scen}_cell_{c}.csv", sep=";")
        df_insert = pd.read_csv(source_insert_path+f"multilabel_data_insert_s{scen}_cell_{c}.csv", sep=";")
        if source_mid_path+f"multilabel_data_original_s{scen}_cell_{c}.csv" in mid_files:
            df_mid = pd.read_csv(source_mid_path+f"multilabel_data_original_s{scen}_cell_{c}.csv", sep=";")
        else:
            df_mid = pd.read_csv(source_none_path+f"multilabel_data_original_s{scen}_cell_{c}.csv", sep=";")
        
        df_final = df_swap.copy()
        selected_cols = [col for col in df_final.columns if col.startswith("Selected")]
        df_final[selected_cols] = (df_swap[selected_cols] + df_insert[selected_cols] + df_mid[selected_cols]).clip(upper=1)
        
        df_final.to_csv(destination_path+f"multilabel_data_s{scen}_cell_{c}.csv", sep=";", index=None)

## entraienement

### singlelabel

In [17]:
def calcul_gap(modele, scaler, cellule, scenarios):
    scenarios_path = f"scenarios/{instance_name}"
    solution_path = f"solution/{instance_name}_upgraded//"
    
    with open(fms_path, 'r') as json_file:
        dic = json.load(json_file)
    s = sys.systeme(dic)
    all_gaps = []
    for f in os.listdir(scenarios_path):
        if int(f.split(".")[0][1:]) not in scenarios:
            continue
        df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f,header=None, index_col=None, sep=";"), nan=0)).astype(int)
        own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
        if len(own_sol_path_list) == 0:
            continue
        sol_path = own_sol_path_list[0]
        sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

        allocators = [Allocation.StaticAllocator(s, sol) for _ in range(len(s.cellules))]
        ref_mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        allocators[cellule] = Allocation.DynamicAllocator(s, Allocation.GlobalSingleLabel(modele, s, cellule), scaler, to_categorical=True)
        mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        gap = (100*(mct - ref_mct)/ref_mct)
        all_gaps.append(gap)

    return np.mean(all_gaps)

#multilabel
def callback(_, trial, cell, X_train, Y_train, scaler, scenarios_test, scores, cv):
    current_params = trial.params
    current_model = RandomForestClassifier(**current_params, random_state=RANDOM_SEED)
    current_model.fit(X_train, Y_train)

    current_gap = calcul_gap(current_model, scaler, cell, scenarios_test)
    #acc = cross_val_score(current_model, X_train, Y_train, cv=cv, scoring=make_scorer(element_wise_accuracy_rf)).mean()
    acc = cross_val_score(current_model, X_train, Y_train, cv=cv, scoring="accuracy").mean()
    #scores.append(acc)
    scores.append([current_gap, acc])
    
#multilabel
def element_wise_accuracy_rf(y_true, y_pred):
    # Convertir en numpy array pour éviter les conflits avec pandas
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float((y_true == y_pred).mean())

#mutlilabel
def objective_rf(trial, X_train, y_train, cv=3, scenarios_test=None, scaler=None, cell=None):
    """
    Fonction objectif pour optimiser les hyperparamètres d'un Random Forest avec Optuna.
    """
    # Définir les hyperparamètres à optimiser
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    # Initialiser le modèle avec les hyperparamètres
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=RANDOM_SEED
    )

    # Effectuer une validation croisée pour évaluer la performance

    model.fit(X_train, y_train)

    return  -calcul_gap(model, scaler, cell, scenarios_test)
    #scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=make_scorer(element_wise_accuracy_rf))
    #scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    #return scores.mean()  # Retourner la moyenne des scores comme métrique


def train_random_forest_with_optuna(X_train, y_train, X_test, y_test, scaler, max_trials=49, alpha=50, beta=0.1, patience_limit=1, random_seed=RANDOM_SEED, cell=1, scenarios_test=[]):
    """
    Entraîner un Random Forest optimisé via une recherche bayésienne sur les hyperparamètres.
    """
    # Initialiser une étude Optuna
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
    total_trials = 0
    remaining_trials = alpha  # Nombre initial de trials
    step = 0
    best_known_score = 0
    patience = 0
    scores = []

    custom_callback = partial(callback, X_train=X_train, Y_train=y_train, scenarios_test=scenarios_test, scores=scores, cell=cell, scaler=scaler, cv=3)

    while total_trials < max_trials and patience < patience_limit:
        print(f"Optimizing: Step {step + 1}, Remaining trials: {remaining_trials}")
        study.optimize(lambda trial: objective_rf(trial, X_train, y_train, cv=3, scenarios_test=scenarios_test, scaler=scaler, cell=cell), n_trials=remaining_trials, callbacks=[custom_callback])
        
        remaining_trials = int(np.ceil(alpha / (1 + beta * step)))
        total_trials += remaining_trials
        step += 1

        best_current_score = study.best_value

        if best_current_score > best_known_score:
            best_known_score = best_current_score
            patience = 0 
        else:
            patience += 1  

    # Afficher les meilleurs paramètres trouvés
    best_params = study.best_params
    print(f"Best Hyperparameters: {best_params}")

    # Entraîner le modèle avec les meilleurs hyperparamètres
    model = RandomForestClassifier(**best_params, random_state=random_seed)
    model.fit(X_train, y_train)

    # Prédire sur le jeu de test et afficher les performances
    predictions = model.predict(X_test)
    print("Optimized Random Forest - Classification Report:")
    print(classification_report(y_test, predictions))


    return model, best_params


In [18]:
with open(fms_path, 'r') as json_file: #data preparation for training
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path = f"generated_data/{instance_name}/single_label/"
test_scenario_count = nombre_de_scenarios//3

file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]


for cell,c in zip(s.cellules,range(len(s.cellules))):
    file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:] if file.endswith(f"cell_{c+1}.csv")])
    kept_cols = cell.header[:-1]
    columns_per_cell.append(kept_cols)
    
    if c == 0:
        test_split = list(range(1, len(file_names_per_cell[0])+1))
        random.shuffle(test_split)
        test_split = test_split[:test_scenario_count]
        test_split_scenarios = test_split[:len(test_split)]
        train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

    paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

    filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
    filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
    

    df_train = pd.concat(filtered_dfs_train, ignore_index=True)
    df_train.fillna(0, inplace=True)


    categorical_cols = [col for col in df_train.columns if col.startswith("Family")]
    to_delete_cols = [col+"_0" for col in df_train.columns if col.startswith("Family")]  
    last_col = [col for col in df_train.columns if col.startswith("Selected")]
    categorical_spec = {
        "Family": [1, 2, 3, 4, 5]
        }
    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_train.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_train[col] == value).astype(float)}))
    df_train = pd.concat([df_train] + new_columns, axis=1)       
    df_train.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_train.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_train.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_train = df_train[colus + [col]]
        colus.append(col)
    
    no_label_cols = [col for col in df_train.columns if not col.startswith("Selected")]
    label_cols = [col for col in df_train.columns if col.startswith("Selected")]

    dfs_train.append(df_train.astype(float))



    df_test = pd.concat(filtered_dfs_test, ignore_index=True)
    df_test.fillna(0, inplace=True)

    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_test.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_test[col] == value).astype(float)}))
    df_test = pd.concat([df_test] + new_columns, axis=1)
    df_test.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_test.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_test.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_test = df_test[colus + [col]]
        colus.append(col)

    dfs_test.append(df_test.astype(float))


### models training

In [19]:
#training (les deux datasets le meme code, changer juste point rouge)

models = []

for i, (train_df, test_df) in enumerate(zip(dfs_train, dfs_test)):

    print(f"\nProcessing dataset {i+1}...")
    nb_classes = len([col for col in train_df.columns if col.startswith("Selected")])
    print(f"\nNumber of ressources {nb_classes}...")

    X_train, y_train = train_df.iloc[:, :-nb_classes], train_df.iloc[:, -nb_classes:].astype(int)
    X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].astype(int)

    model = train_random_forest_with_optuna(X_train, y_train, X_test, y_test, None, max_trials = 500, alpha= 100, beta = 0.1, random_seed=RANDOM_SEED, scenarios_test=test_split_scenarios,cell=i)
    
    predictions_test = model[0].predict(X_test)
    print(f"Validation Performance for dataset {i+1}:\n",classification_report(y_test, predictions_test))
    print(f"- - - saving - - -")
    models.append(model[0])
    with open(f"generated_models/{instance_name}/singlelabel/models_cell{i+1}/standard_RandomForest.pkl", 'wb') as file: #change G for K0
        pickle.dump(model[0], file)
    

[I 2025-07-18 05:46:26,003] A new study created in memory with name: no-name-f9dfc66d-4f45-4058-b4a8-45874c42bc72



Processing dataset 1...

Number of ressources 1...
Optimizing: Step 1, Remaining trials: 100


[I 2025-07-18 05:46:32,097] Trial 0 finished with value: -31.738463090388812 and parameters: {'n_estimators': 144, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: -31.738463090388812.
[I 2025-07-18 05:46:49,233] Trial 1 finished with value: -23.227716140554875 and parameters: {'n_estimators': 267, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -23.227716140554875.
[I 2025-07-18 05:47:07,247] Trial 2 finished with value: -41.13198372310903 and parameters: {'n_estimators': 95, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 1 with value: -23.227716140554875.
[I 2025-07-18 05:47:20,985] Trial 3 finished with value: -28.261872786236346 and parameters: {'n_estimators': 85, 'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 1 with value: -23.2277161

Best Hyperparameters: {'n_estimators': 241, 'max_depth': 20, 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_features': None}


[I 2025-07-18 06:22:53,193] A new study created in memory with name: no-name-8da0d8d6-03cd-4b33-991e-383374a06bf3


...

Best Hyperparameters: {'n_estimators': 283, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


[I 2025-07-18 07:56:04,888] A new study created in memory with name: no-name-43b307d3-c8b2-4e21-a33b-fc0306f76c7a


Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.37      0.39      0.38       449
           1       0.35      0.23      0.28       414
           2       0.35      0.45      0.39       437

    accuracy                           0.36      1300
   macro avg       0.36      0.36      0.35      1300
weighted avg       0.36      0.36      0.35      1300

Validation Performance for dataset 4:
               precision    recall  f1-score   support

           0       0.37      0.39      0.38       449
           1       0.35      0.23      0.28       414
           2       0.35      0.45      0.39       437

    accuracy                           0.36      1300
   macro avg       0.36      0.36      0.35      1300
weighted avg       0.36      0.36      0.35      1300

- - - saving - - -

Processing dataset 5...

Number of ressources 1...
Optimizing: Step 1, Remaining trials: 100


[I 2025-07-18 07:56:11,176] Trial 0 finished with value: -0.33719273487735824 and parameters: {'n_estimators': 144, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.33719273487735824.
[I 2025-07-18 07:56:26,977] Trial 1 finished with value: -0.3106619274283918 and parameters: {'n_estimators': 267, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -0.3106619274283918.
[I 2025-07-18 07:56:42,512] Trial 2 finished with value: -0.32720508554925254 and parameters: {'n_estimators': 95, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 1 with value: -0.3106619274283918.
[I 2025-07-18 07:56:54,366] Trial 3 finished with value: -0.31676056731558017 and parameters: {'n_estimators': 85, 'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 1 with value: -0.310

Best Hyperparameters: {'n_estimators': 124, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': 'sqrt'}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.80      0.80       499
           1       0.88      0.87      0.87       801

    accuracy                           0.84      1300
   macro avg       0.83      0.84      0.83      1300
weighted avg       0.84      0.84      0.84      1300

Validation Performance for dataset 5:
               precision    recall  f1-score   support

           0       0.79      0.80      0.80       499
           1       0.88      0.87      0.87       801

    accuracy                           0.84      1300
   macro avg       0.83      0.84      0.83      1300
weighted avg       0.84      0.84      0.84      1300

- - - saving - - -


## Simulation

In [20]:
with open(fms_path, 'r') as json_file: #5C14R5F #3C7R5F
    dic = json.load(json_file)
s = sys.systeme(dic)

models= []
model_classes = []
for i in range(1,1+nombre_de_cellules):
    with open(f"generated_models/{instance_name}/singlelabel/models_cell{i}/standard_RandomForest.pkl", 'rb') as f:
        m = pickle.load(f)
        models.append(m)
        model_classes.append(Allocation.GlobalSingleLabel(m, s, i))

scenarios_path = f"scenarios/{instance_name}"
solution_path = f"solution/{instance_name}_upgraded/"

results = np.zeros((0,4))
reference_results = np.zeros((0,4))

for f in sorted(os.listdir(scenarios_path), key=lambda y: int(y.split(".")[0][1:])):
    #print(f"{f.split(".")[0]}\t:\t", end = "")
    #if int(f.split(".")[0][1:]) not in test_split_scenarios:
    #    print("dans les scenarios d'entrainement",end="\r")
    #    continue
    df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f, index_col=None, header=None, sep=";"), nan=0)).astype(int)
    own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
    
    sol_path = own_sol_path_list[0]
    sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

    #random_allocs = [Allocation.RandomAllocator(s) for _ in range(nombre_de_cellules)]

    dyn_allocs = [Allocation.DynamicAllocator(s, model_class, None, keep_cols=None, to_categorical=True, singlelabel=True) for model_class in model_classes]
    
    static_allocators = [Allocation.StaticAllocator(s, sol) for _ in range(nombre_de_cellules)]
    
    allocators = [#static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2]
                  ] if instance_name == "K0" else [
                      #static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2], 
                  #static_allocators[3], 
                  dyn_allocs[3], 
                  #static_allocators[4], 
                  dyn_allocs[4]
                  ]
    
    sim = simulation(system=s, scenario=df, allocators=allocators, labels_encoded=False)
    results = np.vstack((results, np.array([f.split(".")[0], sim.average_flowtime(), sim.mean_completion_time(), sim.total_decision_times()])))

    sim_ref = simulation(system=s, scenario=df, allocators=static_allocators)
    reference_results = np.vstack((reference_results, np.array([f.split(".")[0], sim_ref.average_flowtime(), sim_ref.mean_completion_time(), sim_ref.total_decision_times()])))
    
    print(f"{f.split(".")[0]}\t{sim.mean_completion_time()}\t{sim_ref.mean_completion_time()}\t{int(f.split(".")[0][1:]) in test_split_scenarios}")
    #print(f"ended with mct : {sim.mean_completion_time()} while the reference mct is {sim_ref.mean_completion_time()}")
    
    #sim_ref.gantt(path=f"gants/gant_{f.split(".")[0]}_ref.png")
    #sim.gantt(path=f"gants/gant_{f.split(".")[0]}_{total_nb_scenarios//3}.png")

df = pd.DataFrame(results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
df_sorted = df.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

df_ref = pd.DataFrame(reference_results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df_ref['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
ref = df_ref.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

print(f"gap {"ag" if isinstance(allocators[0], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[1], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[2], Allocation.StaticAllocator) else "m"} {(100*(df_sorted - ref)/ref).mean()} %")

s1	347.72	296.5	False
s2	353.07	305.71	False
s3	339.13	303.49	False
s4	333.85	294.63	True
s5	354.78	296.63	True
s6	355.68	297.42	False
s7	357.45	298.24	False
s8	357.28	301.53	False
s9	360.38	312.66	False
s10	351.38	299.32	True
s11	364.18	303.6	True
s12	355.56	309.91	True
s13	320.5	289.79	False
s14	308.95	286.18	False
s15	345.93	290.94	False
s16	341.84	294.05	False
s17	332.88	312.46	False
s18	349.49	302.11	False
s19	325.43	311.99	False
s20	336.87	293.85	False
s21	305.11	284.78	False
s22	381.45	306.75	True
s23	323.44	299.14	False
s24	312.32	292.74	True
s25	326.49	294.01	True
s26	309.22	280.28	True
s27	347.71	301.28	False
s28	340.89	290.57	False
s29	320.02	290.09	True
s30	321.54	287.15	False
s31	334.89	286.95	True
s32	321.65	292.29	False
s33	318.35	284.18	True
s34	334.35	299.02	False
s35	341.7	294.45	False
s36	348.65	287.62	False
s37	317.46	277.46	True
s38	341.95	294.8	False
s39	363.47	310.03	False
s40	312.13	288.4	False
gap m m m 14.099962116169289 %


## Evaluation

In [ ]:
with open(fms_path, 'r') as json_file: #data preparation for training
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path = f"generated_data/{instance_name}/single_label/"
total_nb_scenarios = 40
test_scenario_count = total_nb_scenarios//3

file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]


for cell,c in zip(s.cellules,range(len(s.cellules))):
    file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:total_nb_scenarios] if file.endswith(f"cell_{c+1}.csv")])
    kept_cols = cell.header[:-1]
    columns_per_cell.append(kept_cols)
    
    if c == 0:
        test_split = list(range(1, len(file_names_per_cell[0])+1))
        random.shuffle(test_split)
        test_split = test_split[:test_scenario_count]
        test_split_scenarios = test_split[:len(test_split)]
        train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

    paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

    filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
    filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
    

    df_train = pd.concat(filtered_dfs_train, ignore_index=True)
    df_train.fillna(0, inplace=True)


    categorical_cols = [col for col in df_train.columns if col.startswith("Family")]
    to_delete_cols = [col+"_0" for col in df_train.columns if col.startswith("Family")]  
    last_col = [col for col in df_train.columns if col.startswith("Selected")]
    categorical_spec = {
        "Family": [1, 2, 3, 4, 5]
        }
    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_train.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_train[col] == value).astype(float)}))
    df_train = pd.concat([df_train] + new_columns, axis=1)       
    df_train.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_train.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_train.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_train = df_train[colus + [col]]
        colus.append(col)
    
    no_label_cols = [col for col in df_train.columns if not col.startswith("Selected")]
    label_cols = [col for col in df_train.columns if col.startswith("Selected")]

    dfs_train.append(df_train.astype(float))



    df_test = pd.concat(filtered_dfs_test, ignore_index=True)
    df_test.fillna(0, inplace=True)

    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_test.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_test[col] == value).astype(float)}))
    df_test = pd.concat([df_test] + new_columns, axis=1)
    df_test.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_test.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_test.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_test = df_test[colus + [col]]
        colus.append(col)

    dfs_test.append(df_test.astype(float))
       
models= []
for i in range(1,len(system.cellules)+1):
    with open(f"generated_models/{instance_name}/singlelabel/models_cell{i}/standard_RandomForest.pkl", 'rb') as f:
        models.append(pickle.load(f))

In [81]:
with open(fms_path, 'r') as json_file: #preparation de donnees
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path = f"generated_data/{instance_name}/multilabel/final/"
total_nb_scenarios = 28 * 3
test_scenario_count = 7

file_names_per_cell, columns_per_cell, dfs_trains, dfs_tests = [],[],[],[]


for cell,c in zip(s.cellules,range(len(s.cellules))):
    file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:total_nb_scenarios] if file.endswith(f"cell_{c+1}.csv")])
    kept_cols = cell.header[:-1]
    columns_per_cell.append(kept_cols)
    
    if c == 0:
        test_split = list(range(1, len(file_names_per_cell[0])+1))
        random.shuffle(test_split)
        test_split = test_split[:test_scenario_count]
        test_split_scenarios = test_split[:len(test_split)]
        train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

    paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

    filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
    
    df_test = pd.concat(filtered_dfs_test, ignore_index=True)
    df_test.fillna(0, inplace=True)

    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_test.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_test[col] == value).astype(float)}))
    df_test = pd.concat([df_test] + new_columns, axis=1)
    df_test.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_test.drop(columns=categorical_cols, inplace=True, errors='ignore')
    last_col = [col for col in df_test.columns if col.startswith("Selected")]
    

    colus = df_test.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_test = df_test[colus + [col]]
        colus.append(col)

    dfs_tests.append(df_test.astype(float))

In [82]:
for i, (model, train_df, test_df, test_df_m) in enumerate(zip(models, dfs_train, dfs_test, dfs_tests)):

    print(f"\nProcessing dataset {i+1}...")
    nb_classes = len([col for col in train_df.columns if col.startswith("Selected")])

    X_train, y_train = train_df.iloc[:, :-nb_classes], train_df.iloc[:, -nb_classes:].astype(int)
    X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].astype(int)

    nb_classes = len([col for col in test_df_m.columns if col.startswith("Selected")])
    X_test_m, y_test_m = test_df_m.iloc[:, :-nb_classes], test_df_m.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)


    predictions_test = model.predict(X_test)
    print(f"Validation Performance for dataset {i+1}:\n",classification_report(y_test, predictions_test))
    
    print("accuracy on multilabel labels : ",end="")
    predictions_test = model.predict(X_test_m)
    print(sum([y_test_m[f"Selected Resource_{p}.0"].iloc[j] for j, p in enumerate(predictions_test)])/predictions_test.shape[0])


Processing dataset 1...
Validation Performance for dataset 1:
               precision    recall  f1-score   support

           0       0.78      0.90      0.84       390
           1       0.83      0.68      0.75       299

    accuracy                           0.80       689
   macro avg       0.81      0.79      0.79       689
weighted avg       0.81      0.80      0.80       689

accuracy on multilabel labels : 0.8505747126436781

Processing dataset 2...
Validation Performance for dataset 2:
               precision    recall  f1-score   support

           0       0.68      0.73      0.70       251
           1       0.65      0.51      0.57       195
           2       0.64      0.70      0.67       243

    accuracy                           0.66       689
   macro avg       0.66      0.65      0.65       689
weighted avg       0.66      0.66      0.65       689

accuracy on multilabel labels : 0.7916666666666666

Processing dataset 3...
Validation Performance for dataset 3: